# 99_run — Black-Litterman 실험 실행

**역할**: bl_config.py에 정의된 모든 실험을 walk-forward로 실행하고 결과를 `results/` 폴더에 저장.

## 실행 순서
1. 데이터 로드
2. Dispatcher 함수 정의 (슬롯별 config → 함수 분기)
3. walk_forward() 함수 정의
4. 전체 실험 실행 → pkl 저장

> 분석/시각화는 **99_analyze.ipynb**에서.

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
import platform
import importlib
from pathlib import Path

import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')

# 모듈 캐시 강제 갱신 (bl_functions, bl_config 변경 시)
import bl_functions, bl_config
importlib.reload(bl_functions)
importlib.reload(bl_config)

from bl_functions import (
    compute_sigma, compute_daily_slice, compute_pi,
    build_P,
    compute_Q_fixed, compute_Q_lambda, compute_Q_inv_lambda, compute_Q_raw_lam, compute_Q_vol_spread,
    compute_Q_ff3_paper, compute_Q_ff3_paper_mean,
    compute_omega_he, compute_omega_rmse,
    black_litterman, optimize_portfolio,
    compute_turnover, apply_tc, compute_metrics,
)
from bl_config import EXPERIMENTS, get_changed_slots

BASE_DIR    = Path.cwd()
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

assert DATA_DIR.exists(), f'data/ 폴더 없음. 01_DataCollection.ipynb 먼저 실행하세요.\n경로: {DATA_DIR}'

# ── 공통 파라미터 ──────────────────────────────────────────────
TRAIN_WINDOW  = 60
THRESH_DAILY  = 0.9
TAU           = 0.1
PCT_GROUP     = 0.30
START_PRED    = '2010-01-01'

print(f'설정 완료')
print(f'  DATA_DIR    : {DATA_DIR}')
print(f'  RESULTS_DIR : {RESULTS_DIR}')
print(f'  실험 수     : {len(EXPERIMENTS)}개')

설정 완료
  DATA_DIR    : c:\workspace\camp\project\finance_project\final\data
  RESULTS_DIR : c:\workspace\camp\project\finance_project\final\results
  실험 수     : 156개


In [2]:
# ── 데이터 로드 ───────────────────────────────────────────────
panel     = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
panel     = panel.set_index(['date', 'ticker'])
daily_ret = pd.read_pickle(DATA_DIR / 'daily_returns.pkl')

all_dates  = panel.index.get_level_values('date').unique().sort_values()
pred_dates = all_dates[all_dates >= START_PRED]

spy_series = panel['spy_ret'].groupby(level='date').first()
rf_series  = panel['rf_1m'].groupby(level='date').first()
ret_pivot  = panel['ret_1m'].unstack('ticker')   # FF3 Q 계산용

# FF3 팩터
import io, re, zipfile, requests
FF3_PATH = DATA_DIR / 'ff3_monthly.csv'
if FF3_PATH.exists():
    ff3 = pd.read_csv(FF3_PATH, index_col='date', parse_dates=True)
else:
    print('FF3 다운로드 중...')
    url  = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip'
    resp = requests.get(url, timeout=60)
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        raw = zf.read(zf.namelist()[0]).decode('utf-8', errors='ignore')
    lines = raw.splitlines()
    start = next(i for i, l in enumerate(lines) if re.match(r'^\s*\d{6}\s*,', l))
    end   = next((i for i in range(start, len(lines)) if not re.match(r'^\s*\d{6}\s*,', lines[i])), len(lines))
    df    = pd.read_csv(io.StringIO('\n'.join(lines[start-1:end])))
    df.columns = [c.strip() for c in df.columns]
    dc = df.columns[0]
    df[dc] = pd.to_datetime(df[dc].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
    ff3 = df.rename(columns={dc:'date','Mkt-RF':'mkt_rf','SMB':'smb','HML':'hml','RF':'rf'}).set_index('date').astype(float) / 100.0
    ff3.to_csv(FF3_PATH)

print(f'패널       : {panel.shape}')
print(f'일별 수익률: {daily_ret.shape}')
print(f'예측 기간  : {pred_dates[0].date()} ~ {pred_dates[-1].date()} ({len(pred_dates)}개월)')

패널       : (103878, 11)
일별 수익률: (5595, 822)
예측 기간  : 2010-01-31 ~ 2025-12-31 (192개월)


In [4]:
# ── LSTM 예측값 로드 (없으면 None) ────────────────────────────
# bl_config.py의 BASELINE['lstm_pred_path']에 경로 지정
# 파일이 없으면 LSTM 관련 실험은 자동 스킵

from bl_config import BASELINE

_lstm_path = Path(BASELINE.get('lstm_pred_path', ''))
if _lstm_path.exists():
    _raw = pd.read_csv(_lstm_path, parse_dates=['date'])
    _raw['vol_pred'] = np.exp(_raw['y_pred_ensemble']) * np.sqrt(252)          # log-RV → 실제 변동성
    _raw['abs_err']  = (_raw['y_pred_ensemble'] - _raw['y_true']).abs()

    # 원본 (date, ticker) 인덱스 — omega_rmse 계산용
    lstm_preds = _raw.set_index(['date', 'ticker'])

    # ── 월별 피벗 (Phase3 get_monthly_pred 방식) ──────────────
    # 각 종목·월 기준 마지막 예측값 → 시장 거래일 월말 매핑
    _m = _raw.copy()
    _m['month'] = _m['date'].dt.to_period('M')
    _last = _m.groupby(['ticker', 'month'])['vol_pred'].last().reset_index()
    _month_map = {pd.Timestamp(d).to_period('M'): pd.Timestamp(d) for d in pred_dates}
    _last['pred_date'] = _last['month'].map(_month_map)
    _last = _last.dropna(subset=['pred_date'])
    lstm_pred_monthly = _last.pivot_table(
        index='pred_date', columns='ticker', values='vol_pred')

    print(f'LSTM 예측 로드: {len(_raw):,}행')
    print(f'LSTM 월별 피벗: {lstm_pred_monthly.shape[0]}개월 × {lstm_pred_monthly.shape[1]}종목')
    LSTM_AVAILABLE = True
else:
    lstm_preds        = None
    lstm_pred_monthly = None
    LSTM_AVAILABLE    = False
    print(f'⚠  LSTM 예측 파일 없음 → lstm 관련 실험 스킵')
    print(f'   경로: {_lstm_path}')
# ── ANN 예측값 로드 (없으면 None) ─────────────────────────────
# bl_config.py의 BASELINE['ann_pred_path']에 경로 지정
# 파일이 없으면 paper_ann 실험은 자동 스킵
_ann_path_str = BASELINE.get('ann_pred_path') or 'data/paper_ann_predictions.csv'
_ann_path = Path(_ann_path_str)
if _ann_path.is_file():
    _ann = pd.read_csv(_ann_path, parse_dates=['date'])
    _ann['vol_pred'] = np.exp(_ann['y_pred_ensemble']) * np.sqrt(252)

    # 월별 피벗 (LSTM 와 동일 구조)
    _ann['month'] = _ann['date'].dt.to_period('M')
    _last_a = _ann.groupby(['ticker', 'month'])['vol_pred'].last().reset_index()
    _month_map_a = {pd.Timestamp(d).to_period('M'): pd.Timestamp(d) for d in pred_dates}
    _last_a['pred_date'] = _last_a['month'].map(_month_map_a)
    _last_a = _last_a.dropna(subset=['pred_date'])
    ann_pred_monthly = _last_a.pivot_table(
        index='pred_date', columns='ticker', values='vol_pred')

    print(f'ANN 예측 로드: {len(_ann):,}행')
    print(f'ANN 월별 피벗: {ann_pred_monthly.shape[0]}개월 × {ann_pred_monthly.shape[1]}종목')
    ANN_AVAILABLE = True
else:
    ann_pred_monthly = None
    ANN_AVAILABLE    = False
    print(f'⚠  ANN 예측 파일 없음 → paper_ann 실험 스킵')
    print(f'   경로: {_ann_path}')


LSTM 예측 로드: 2,468,883행
LSTM 월별 피벗: 192개월 × 613종목
ANN 예측 로드: 64,628행
ANN 월별 피벗: 180개월 × 503종목


In [5]:
# ══════════════════════════════════════════════════════════════
# Dispatcher 함수 (config → 실제 함수 호출)
# 새 방식 추가 시 해당 elif만 추가
# ══════════════════════════════════════════════════════════════

def get_vol_series(cfg, month_df, pred_date):
    """P 행렬에 쓸 변동성 시리즈 반환."""
    mode = cfg.get('p_mode', 'trailing_vol21')

    if mode == 'trailing_vol21':
        return month_df['vol_21d']

    elif mode == 'trailing_vol252':
        return month_df['vol_252d']

    elif mode == 'lstm_predicted':
        if lstm_pred_monthly is None or pred_date not in lstm_pred_monthly.index:
            return month_df['vol_21d']
        pred_slice = lstm_pred_monthly.loc[pred_date].dropna()
        vol = month_df['vol_21d'].copy()
        common = vol.index.intersection(pred_slice.index)
        if len(common) >= 5:
            vol[common] = pred_slice[common]
        return vol

    elif mode == 'ann_predicted':
        if ann_pred_monthly is None or pred_date not in ann_pred_monthly.index:
            return month_df['vol_21d']
        pred_slice = ann_pred_monthly.loc[pred_date].dropna()
        vol = month_df['vol_21d'].copy()
        common = vol.index.intersection(pred_slice.index)
        if len(common) >= 5:
            vol[common] = pred_slice[common]
        return vol

    else:
        raise ValueError(f'p_mode={mode!r} 미지원')


def get_prior_weights(cfg, valid_tix, mcap, vol=None):
    """prior 시장가중치 w_mkt 반환. capm_rp는 vol(trailing) 필요."""
    prior = cfg.get('prior', 'capm_mcap')
    if prior == 'capm_mcap':
        return (mcap / mcap.sum()).reindex(valid_tix).fillna(0)
    elif prior == 'capm_eq':
        n = len(valid_tix)
        return pd.Series(1.0 / n, index=valid_tix)
    elif prior == 'capm_rp':
        # Risk Parity: 1/σ 정규화 (자산 독립 가정, 단순 RP)
        if vol is None:
            raise ValueError('capm_rp requires vol parameter')
        inv_vol = 1.0 / vol.replace(0, np.nan).dropna()
        w = inv_vol / inv_vol.sum()
        return w.reindex(valid_tix).fillna(0)
    else:
        raise ValueError(f'prior={prior!r} 미지원')


def get_Q(cfg, P, valid_tix, train_dates, pred_date, all_d, lam=2.5, spy_excess=0.0, sigma2_mkt=0.001):
    """Q 값 반환. q_mode='capm'/'none'은 walk_forward에서 직접 처리."""
    mode = cfg.get('q_mode', 'fixed')

    if mode == 'fixed':
        return compute_Q_fixed(cfg.get('q_value', 0.003))

    elif mode == 'ff3_paper':
        # 기존 구현: X_next = 직전월 실현 팩터 (random walk)
        # Q만 반환, Ω는 walk-forward 루프에서 전월 오차²로 계산
        thresh_m    = int(len(train_dates) * 0.7)
        monthly_sl  = ret_pivot.reindex(index=train_dates, columns=valid_tix).dropna(axis=1, thresh=thresh_m)
        ff3_train   = ff3.reindex(train_dates)
        rf_train    = rf_series.reindex(train_dates)
        return compute_Q_ff3_paper(P.reindex(monthly_sl.columns).fillna(0), monthly_sl, ff3_train, rf_train)

    elif mode == 'ff3_paper_mean':
        # 학계 표준 변형: X_next = 60개월 팩터 평균 (mean reversion)
        # Fama-MacBeth (1973) / Cochrane (2005) 표준. random walk 대비 노이즈 작음.
        thresh_m    = int(len(train_dates) * 0.7)
        monthly_sl  = ret_pivot.reindex(index=train_dates, columns=valid_tix).dropna(axis=1, thresh=thresh_m)
        ff3_train   = ff3.reindex(train_dates)
        rf_train    = rf_series.reindex(train_dates)
        return compute_Q_ff3_paper_mean(P.reindex(monthly_sl.columns).fillna(0), monthly_sl, ff3_train, rf_train)

    elif mode == 'lambda':
        return compute_Q_lambda(lam, cfg.get('q_value', 0.003), cfg.get('lam_mean', 2.5))

    elif mode == 'inv_lambda':
        return compute_Q_inv_lambda(lam, cfg.get('q_value', 0.003), cfg.get('lam_mean', 2.5))

    elif mode == 'raw_lam':
        return compute_Q_raw_lam(spy_excess, sigma2_mkt, cfg.get('q_value', 0.003), cfg.get('lam_mean', 2.5))

    elif mode == 'vol_spread':
        # LSTM 예측 vol 격차 기반 동적 Q (사용자 추가)
        if lstm_pred_monthly is None or pred_date not in lstm_pred_monthly.index:
            return cfg.get('q_value', 0.003)  # LSTM 없으면 fallback to fixed

        # 현재 시점 예측 vol
        vol_pred_curr = lstm_pred_monthly.loc[pred_date].dropna()

        # 과거 spread들 (train_dates 구간) — look-ahead bias 없음
        past_spreads = []
        for past_dt in train_dates:
            if past_dt not in lstm_pred_monthly.index:
                continue
            vp = lstm_pred_monthly.loc[past_dt].dropna()
            if len(vp) < 20:
                continue
            n_grp = max(1, int(len(vp) * PCT_GROUP))
            sorted_vp = vp.sort_values()
            low_m  = sorted_vp.iloc[:n_grp].mean()
            high_m = sorted_vp.iloc[-n_grp:].mean()
            past_spreads.append(high_m - low_m)
        if not past_spreads:
            return cfg.get('q_value', 0.003)
        spread_ref = float(np.median(past_spreads))

        return compute_Q_vol_spread(
            P, vol_pred_curr,
            cfg.get('q_value', 0.003),
            spread_ref,
        )

    else:
        raise ValueError(f'q_mode={mode!r} 미지원')


# abs_err 실제 중앙값 (~0.24) — inf 값 존재하므로 median 사용
_OMEGA_BASE_RMSE = 0.24

def get_omega(cfg, P, Sigma, pred_date):
    """Omega 값 반환."""
    mode = cfg.get('omega_mode', 'he_litterman')

    if mode == 'he_litterman':
        return compute_omega_he(P, Sigma, TAU)

    elif mode == 'rmse':
        if lstm_preds is None:
            return compute_omega_he(P, Sigma, TAU)
        cutoff = pred_date - pd.DateOffset(months=12)
        recent = lstm_preds[
            (lstm_preds.index.get_level_values('date') >= cutoff) &
            (lstm_preds.index.get_level_values('date') <= pred_date)
        ]
        if len(recent) > 0:
            # inf/nan 필터링 후 median 사용 (mean은 inf 오염 가능)
            err_vals = recent['abs_err'].replace([np.inf, -np.inf], np.nan).dropna()
            pred_rmse = float(err_vals.median()) if len(err_vals) > 0 else _OMEGA_BASE_RMSE
        else:
            pred_rmse = _OMEGA_BASE_RMSE
        return compute_omega_rmse(P, Sigma, TAU, pred_rmse, base_rmse=_OMEGA_BASE_RMSE)

    # 'ff3_paper' 모드는 walk_forward에서 직접 처리 (직전월 예측오차²) — get_omega로 안 옴

    else:
        raise ValueError(f'omega_mode={mode!r} 미지원')


print('Dispatcher 함수 정의 완료')

Dispatcher 함수 정의 완료


In [ ]:
import time

# ══════════════════════════════════════════════════════════════
# 월별 공유 데이터 캐시 (모든 실험에서 동일한 부분 — 한 번만 계산)
#   Sigma (LedoitWolf), month_df, mcap, train_dates, spy_excess, sigma2_mkt
#   실험별로 달라지는 것: vol_series(p_mode), P(p_weight), Q, omega, w
# ══════════════════════════════════════════════════════════════

monthly_cache = {}
_cache_t0 = time.time()

for i, pred_date in enumerate(pred_dates):
    if i % 24 == 0:
        elapsed = time.time() - _cache_t0
        pct     = (i + 1) / len(pred_dates)
        eta     = elapsed / pct * (1 - pct) if pct > 0.01 else 0
        print(f'  캐시 {pred_date.strftime("%Y-%m")} ({i+1}/{len(pred_dates)}, {pct:.0%}) | '
              f'경과 {elapsed/60:.1f}분 | ETA {eta/60:.1f}분', flush=True)
    try:
        month_df = panel.xs(pred_date, level='date').dropna(
            subset=['vol_21d', 'log_mcap', 'ret_1m'])
        if len(month_df) < 30:
            continue

        daily_slice, valid_tix = compute_daily_slice(
            pred_date, month_df.index.tolist(),
            daily_ret, TRAIN_WINDOW, THRESH_DAILY)
        if len(valid_tix) < 20:
            continue

        Sigma    = compute_sigma(daily_slice, scale=21)
        month_df = month_df.reindex(valid_tix)
        mcap     = np.exp(month_df['log_mcap'])

        idx         = all_dates.get_loc(pred_date)
        train_dates = all_dates[max(0, idx - TRAIN_WINDOW): idx]
        spy_s       = spy_series.reindex(train_dates)
        rf_s        = rf_series.reindex(train_dates)

        next_date = all_dates[idx + 1] if idx + 1 < len(all_dates) else None

        monthly_cache[pred_date] = {
            'month_df'   : month_df,
            'valid_tix'  : valid_tix,
            'Sigma'      : Sigma,
            'mcap'       : mcap,
            'train_dates': train_dates,
            'spy_excess' : float((spy_s - rf_s).mean()),
            'sigma2_mkt' : float(spy_s.var()),
            'next_date'  : next_date,
        }
    except Exception as e:
        print(f'  [캐시 에러] {pred_date.date()}: {e}')

_cache_min = (time.time() - _cache_t0) / 60
print(f'\n캐시 완료: {len(monthly_cache)}개월 / {len(pred_dates)}개월 | {_cache_min:.1f}분 소요')
print('이후 실험은 Sigma 재계산 없이 캐시에서 로드')

  캐시 2010-01 (1/180, 1%) | 경과 0.0분 | ETA 0.0분
  캐시 2012-01 (25/180, 14%) | 경과 0.0분 | ETA 0.3분
  캐시 2014-01 (49/180, 27%) | 경과 0.1분 | ETA 0.3분
  캐시 2016-01 (73/180, 41%) | 경과 0.2분 | ETA 0.2분
  캐시 2018-01 (97/180, 54%) | 경과 0.2분 | ETA 0.2분
  캐시 2020-01 (121/180, 67%) | 경과 0.3분 | ETA 0.1분
  캐시 2022-01 (145/180, 81%) | 경과 0.3분 | ETA 0.1분
  캐시 2024-01 (169/180, 94%) | 경과 0.4분 | ETA 0.0분

캐시 완료: 180개월 / 180개월 | 0.4분 소요
이후 실험은 Sigma 재계산 없이 캐시에서 로드


In [ ]:
import time

# ══════════════════════════════════════════════════════════════
# walk_forward(cfg) — 실험 하나 실행 (캐시에서 Sigma 로드)
# ══════════════════════════════════════════════════════════════

def walk_forward(cfg: dict) -> dict:
    name       = cfg['name']
    tc         = cfg.get('tc', 0.001)
    max_w      = cfg.get('max_weight', 0.10)
    q_mode     = cfg.get('q_mode', 'fixed')
    p_weight   = cfg.get('p_weight', 'mcap')
    is_naive   = (q_mode == 'none')
    is_capm    = (q_mode == 'capm')

    ret_list, comp_list, meta_list = [], [], []
    spy_list, err_list = [], []
    weights_history = {}
    prev_w = None
    # omega_paper 상태 (전월 Q 예측 이력)
    _op_q_prev = None   # 직전 월 Q 예측값
    _op_P_prev = None   # 직전 월 P 벡터
    _op_errors = []     # 누적 예측 오차
    _t0 = time.time()

    for i, pred_date in enumerate(pred_dates):

        if i % 12 == 0 or i == len(pred_dates) - 1:
            elapsed = time.time() - _t0
            pct     = (i + 1) / len(pred_dates)
            eta     = elapsed / pct * (1 - pct) if pct > 0.01 else 0
            print(f'  [{name}] {pred_date.strftime("%Y-%m")} '
                  f'({i+1}/{len(pred_dates)}, {pct:.0%}) | '
                  f'경과 {elapsed/60:.1f}분 | ETA {eta/60:.1f}분', flush=True)

        if pred_date not in monthly_cache:
            continue
        c           = monthly_cache[pred_date]
        month_df    = c['month_df']
        valid_tix   = c['valid_tix']
        Sigma       = c['Sigma']
        mcap        = c['mcap']
        train_dates = c['train_dates']
        spy_excess  = c['spy_excess']
        sigma2_mkt  = c['sigma2_mkt']
        next_date   = c['next_date']

        try:
            w_mkt   = get_prior_weights(cfg, valid_tix, mcap, vol=month_df['vol_21d'])
            pi, lam = compute_pi(Sigma, w_mkt, spy_excess, sigma2_mkt)

            vol_series = get_vol_series(cfg, month_df, pred_date)
            P = build_P(
                vol_series.reindex(valid_tix).fillna(vol_series.median()),
                mcap, pct=PCT_GROUP, weighting=p_weight)

            if is_capm:
                w = optimize_portfolio(pi, Sigma.values, lam, max_w)
                mu_meta = None

            elif is_naive:
                n_g     = max(1, int(len(valid_tix) * PCT_GROUP))
                vol_v   = vol_series.reindex(valid_tix)
                low_tix = vol_v.sort_values().index[:n_g]

                if p_weight == 'mcap':
                    w_low = mcap.reindex(low_tix)
                    w_low = w_low / w_low.sum()
                elif p_weight == 'rp':
                    inv_v = (1.0 / vol_v[low_tix]).replace(0, np.nan).dropna()
                    w_low = inv_v / inv_v.sum()
                else:
                    w_low = pd.Series(1.0 / len(low_tix), index=low_tix)

                w = w_low.reindex(valid_tix).fillna(0)
                w = w.clip(upper=max_w)
                w = w / w.sum()
                mu_meta = None

            else:
                lam_q = float(np.clip(spy_excess / sigma2_mkt, 0.5, 10.0)) if sigma2_mkt > 1e-10 else 2.5
                Q     = get_Q(cfg, P, valid_tix, train_dates, pred_date, all_dates,
                              lam=lam_q, spy_excess=spy_excess, sigma2_mkt=sigma2_mkt)
                # ff3_paper: Q, Omega 동시 반환
                if cfg.get("omega_mode") == "ff3_paper":
                    # 논문 방식: 직전 월 예측오차²을 Omega로 사용
                    if _op_q_prev is not None and _op_P_prev is not None:
                        actual_p = float(
                            month_df["ret_1m"].reindex(_op_P_prev.index).fillna(0)
                            @ _op_P_prev
                        )
                        omega = max((_op_q_prev - actual_p) ** 2, 1e-8)
                    else:
                        omega = compute_omega_he(P, Sigma, TAU)  # 첫달 폴백
                    _op_q_prev = Q
                    _op_P_prev = P.copy()
                else:
                    omega = get_omega(cfg, P, Sigma, pred_date)
                mu_BL, sig_BL = black_litterman(pi, Sigma, P, Q, omega, TAU)
                w      = optimize_portfolio(mu_BL, Sigma.values, lam, max_w)
                mu_meta = Q

            actual_ret = month_df['fwd_ret_1m'].reindex(valid_tix).fillna(0)
            gross_ret  = float(w @ actual_ret)
            turnover   = compute_turnover(w, prev_w) if prev_w is not None else 0.0
            net_ret    = apply_tc(gross_ret, turnover, tc)
            r_spy      = float(spy_series.get(next_date, np.nan)) if next_date else np.nan

            ret_list.append({'date': pred_date, 'ret': net_ret, 'gross_ret': gross_ret})
            spy_list.append({'date': pred_date, 'ret': r_spy})

            vol_col      = month_df['vol_21d'].reindex(valid_tix)
            n_g          = max(1, int(len(valid_tix) * PCT_GROUP))
            sv           = vol_col.sort_values()
            low_tix_all  = sv.index[:n_g].tolist()
            high_tix_all = sv.index[-n_g:].tolist()

            comp_list.append({
                'date'        : pred_date,
                'n_stocks'    : len(valid_tix),
                'eff_n'       : 1.0 / float((w**2).sum()) if (w**2).sum() > 0 else 0,
                'top10_share' : float(w.nlargest(10).sum()),
                'top1_weight' : float(w.max()),
                'top1_ticker' : w.idxmax(),
                'avg_vol'     : float((w * vol_col).sum()),
                'low_weight'  : float(w.reindex(low_tix_all).sum()),
                'high_weight' : float(w.reindex(high_tix_all).sum()),
                'turnover'    : turnover,
                'tc_cost'     : turnover * tc,
            })

            meta_list.append({'date': pred_date, 'Q': mu_meta, 'lam': lam})
            weights_history[pred_date] = w
            prev_w = w

        except Exception as e:
            err_list.append({'date': pred_date, 'error': str(e)})
            if len(err_list) <= 3:
                print(f'  [{name}] 에러 {pred_date.date()}: {e}')

    total_min = (time.time() - _t0) / 60
    ret_df  = pd.DataFrame(ret_list).set_index('date')  if ret_list  else pd.DataFrame()
    spy_df  = pd.DataFrame(spy_list).set_index('date')  if spy_list  else pd.DataFrame()
    comp_df = pd.DataFrame(comp_list).set_index('date') if comp_list else pd.DataFrame()
    meta_df = pd.DataFrame(meta_list).set_index('date') if meta_list else pd.DataFrame()
    wts_df  = pd.DataFrame(weights_history).T            if weights_history else pd.DataFrame()

    print(f'  → [{name}] 완료: {len(ret_df)}개월 성공 / {len(err_list)}개 에러 / {total_min:.1f}분 소요')

    return {
        'config'   : cfg,
        'ret'      : ret_df['ret']       if 'ret'       in ret_df.columns else pd.Series(dtype=float),
        'gross_ret': ret_df['gross_ret'] if 'gross_ret' in ret_df.columns else pd.Series(dtype=float),
        'spy_ret'  : spy_df['ret']       if 'ret'       in spy_df.columns else pd.Series(dtype=float),
        'weights'  : wts_df,
        'comp'     : comp_df,
        'meta'     : meta_df,
        'errors'   : err_list,
    }


print('walk_forward() 함수 정의 완료 (캐시 사용, Σ_BL 적용)')

walk_forward() 함수 정의 완료 (캐시 사용, Σ_BL 적용)


In [ ]:
# ══════════════════════════════════════════════════════════════
# 전체 실험 실행 + 저장 — joblib threading 병렬 (4 worker)
# ══════════════════════════════════════════════════════════════
# threading 안정성: walk_forward 가 monthly_cache 를 read-only 로만 사용,
# 각 cfg 는 독립, pkl 저장은 다른 파일 → race condition 없음.
# scipy.optimize (SLSQP) 와 numpy BLAS 가 GIL release 하므로 4 worker 효과적.

from joblib import Parallel, delayed

# ── 팀 분할 (round-robin) ─────────────────────────
TEAM_RANK = 0   # ⚠ 본인 = 0, 팀원 B/C/D = 1/2/3
TEAM_SIZE = 1
EXPERIMENTS = [e for i, e in enumerate(EXPERIMENTS) if i % TEAM_SIZE == TEAM_RANK]
print(f'⚙ Team rank {TEAM_RANK}/{TEAM_SIZE}: {len(EXPERIMENTS)} 실험 할당')

SKIP_IF_EXISTS = True   # True → 이미 저장된 실험은 재실행 않고 스킵
N_JOBS         = 1      # 병렬 worker 수 (CPU core 수 / 메모리 고려하여 조정)
TEST_MODE      = False  # True 면 sample TEST_N 실험만 실행 (디버그용). False 가 본 실행
TEST_N         = 3

# 실행 대상 결정 (skip 사유 사전 필터링)
def _should_skip(cfg):
    name = cfg['name']
    out_path = RESULTS_DIR / f'{name}.pkl'
    if cfg.get('p_mode') == 'lstm_predicted' and not LSTM_AVAILABLE:
        return f'[SKIP] {name} — LSTM 예측 파일 없음'
    if cfg.get('p_mode') == 'ann_predicted' and not ANN_AVAILABLE:
        return f'[SKIP] {name} — ANN 예측 파일 없음'
    if cfg.get('omega_mode') == 'rmse' and not LSTM_AVAILABLE:
        return f'[SKIP] {name} — LSTM 예측 파일 없음'
    if SKIP_IF_EXISTS and out_path.exists():
        return f'[SKIP] {name} — 이미 저장됨'
    return None  # 실행 대상

run_list, skipped = [], []
for cfg in EXPERIMENTS:
    skip_reason = _should_skip(cfg)
    if skip_reason is None:
        run_list.append(cfg)
    else:
        skipped.append(cfg['name'])
        print(skip_reason)

if TEST_MODE:
    run_list = run_list[:TEST_N]
    print(f'\n⚠️  TEST_MODE — 처음 {TEST_N} 실험만 실행')

print(f'\n실행 예정: {len(run_list)}개 / 전체 {len(EXPERIMENTS)}개  (병렬 worker: {N_JOBS})')
print(f'스킵: {len(skipped)}개\n')


def _run_one(cfg):
    """worker 함수 — walk_forward 후 pkl 저장."""
    name = cfg['name']
    out_path = RESULTS_DIR / f'{name}.pkl'
    t0 = time.time()
    try:
        result = walk_forward(cfg)
        with open(out_path, 'wb') as f:
            pickle.dump(result, f)
        elapsed = time.time() - t0
        return ('OK', name, elapsed)
    except Exception as e:
        return ('ERR', name, str(e))


_all_t0 = time.time()

# joblib threading 병렬 실행
# verbose=10: 매 시점마다 진행 표시
results = Parallel(n_jobs=N_JOBS, backend='threading', verbose=10)(
    delayed(_run_one)(cfg) for cfg in run_list
)

# 결과 집계
completed, errors = [], []
for status, name, info in results:
    if status == 'OK':
        completed.append((name, info))
        print(f'  ✓ {name}  ({info/60:.1f}분)')
    else:
        errors.append((name, info))
        print(f'  ✗ {name}  ERROR: {info[:100]}')

total_elapsed = (time.time() - _all_t0) / 60
print(f'\n{"="*60}')
print(f'완료: {len(completed)}개 / 에러: {len(errors)}개 / 스킵: {len(skipped)}개')
print(f'총 소요: {total_elapsed:.1f}분 (병렬 {N_JOBS} worker)')
print(f'저장 위치: {RESULTS_DIR}')
if errors:
    print(f'\n⚠ 에러 발생 실험:')
    for name, msg in errors:
        print(f'  {name}: {msg[:100]}')


⚙ Team rank 0/1: 233 실험 할당
[SKIP] baseline — 이미 저장됨
[SKIP] capm_no_bl — 이미 저장됨
[SKIP] naive_lowvol — 이미 저장됨
[SKIP] prior_eq — 이미 저장됨
[SKIP] prior_rp — 이미 저장됨
[SKIP] p_eq — 이미 저장됨
[SKIP] p_rp — 이미 저장됨
[SKIP] p_vol_mcap — 이미 저장됨
[SKIP] q_lambda — 이미 저장됨
[SKIP] q_inv_lambda — 이미 저장됨
[SKIP] q_raw_lam — 이미 저장됨
[SKIP] prior_eq_q_lambda — 이미 저장됨
[SKIP] prior_eq_q_raw_lam — 이미 저장됨
[SKIP] omega_paper — 이미 저장됨
[SKIP] omega_rmse — 이미 저장됨
[SKIP] q_ff3_paper — 이미 저장됨
[SKIP] q_ff3_paper_omega_paper — 이미 저장됨
[SKIP] q_ff3_paper_mean — 이미 저장됨
[SKIP] q_ff3_paper_mean_omega_paper — 이미 저장됨
[SKIP] paper_lstm — 이미 저장됨
[SKIP] paper_ann — 이미 저장됨
[SKIP] p_lstm_vol_mcap — 이미 저장됨
[SKIP] baseline_q55 — 이미 저장됨
[SKIP] baseline_q64 — 이미 저장됨
[SKIP] baseline_q70 — 이미 저장됨
[SKIP] mat_mcap_eq_fpm_pap — 이미 저장됨
[SKIP] mat_mcap_rp_fpm_pap — 이미 저장됨
[SKIP] mat_mcap_eq_fpm_pap_ann — 이미 저장됨
[SKIP] mat_mcap_rp_fpm_pap_ann — 이미 저장됨
[SKIP] mat_eq_mcap_fpm_pap — 이미 저장됨
[SKIP] mat_rp_mcap_fpm_pap — 이미 저장됨
[SKIP] mat_eq_mcap_fpm_pap_

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:  2.7min


  [mat_eq_eq_fpm_pap] 2011-01 (13/180, 7%) | 경과 0.1분 | ETA 0.9분
  [mat_eq_eq_fpm_pap] 2012-01 (25/180, 14%) | 경과 0.2분 | ETA 1.2분
  [mat_eq_eq_fpm_pap] 2013-01 (37/180, 21%) | 경과 0.4분 | ETA 1.5분
  [mat_eq_eq_fpm_pap] 2014-01 (49/180, 27%) | 경과 0.5분 | ETA 1.4분
  [mat_eq_eq_fpm_pap] 2015-01 (61/180, 34%) | 경과 0.6분 | ETA 1.1분
  [mat_eq_eq_fpm_pap] 2016-01 (73/180, 41%) | 경과 0.8분 | ETA 1.2분
  [mat_eq_eq_fpm_pap] 2017-01 (85/180, 47%) | 경과 1.0분 | ETA 1.1분
  [mat_eq_eq_fpm_pap] 2018-01 (97/180, 54%) | 경과 1.1분 | ETA 0.9분
  [mat_eq_eq_fpm_pap] 2019-01 (109/180, 61%) | 경과 1.2분 | ETA 0.8분
  [mat_eq_eq_fpm_pap] 2020-01 (121/180, 67%) | 경과 1.3분 | ETA 0.6분
  [mat_eq_eq_fpm_pap] 2021-01 (133/180, 74%) | 경과 1.5분 | ETA 0.5분
  [mat_eq_eq_fpm_pap] 2022-01 (145/180, 81%) | 경과 1.6분 | ETA 0.4분
  [mat_eq_eq_fpm_pap] 2023-01 (157/180, 87%) | 경과 1.7분 | ETA 0.3분
  [mat_eq_eq_fpm_pap] 2024-01 (169/180, 94%) | 경과 1.9분 | ETA 0.1분
  [mat_eq_eq_fpm_pap] 2024-12 (180/180, 100%) | 경과 2.0분 | ETA 0.0분
  → [mat_eq_eq_fpm

[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:  8.2min


  [mat_rp_eq_fpm_pap_ann] 2011-01 (13/180, 7%) | 경과 0.1분 | ETA 0.9분
  [mat_rp_eq_fpm_pap_ann] 2012-01 (25/180, 14%) | 경과 0.2분 | ETA 1.1분
  [mat_rp_eq_fpm_pap_ann] 2013-01 (37/180, 21%) | 경과 0.4분 | ETA 1.4분
  [mat_rp_eq_fpm_pap_ann] 2014-01 (49/180, 27%) | 경과 0.4분 | ETA 1.2분
  [mat_rp_eq_fpm_pap_ann] 2015-01 (61/180, 34%) | 경과 0.5분 | ETA 0.9분
  [mat_rp_eq_fpm_pap_ann] 2016-01 (73/180, 41%) | 경과 0.7분 | ETA 1.1분
  [mat_rp_eq_fpm_pap_ann] 2017-01 (85/180, 47%) | 경과 0.9분 | ETA 1.1분
  [mat_rp_eq_fpm_pap_ann] 2018-01 (97/180, 54%) | 경과 1.1분 | ETA 0.9분
  [mat_rp_eq_fpm_pap_ann] 2019-01 (109/180, 61%) | 경과 1.5분 | ETA 1.0분
  [mat_rp_eq_fpm_pap_ann] 2020-01 (121/180, 67%) | 경과 1.6분 | ETA 0.8분
  [mat_rp_eq_fpm_pap_ann] 2021-01 (133/180, 74%) | 경과 1.9분 | ETA 0.7분
  [mat_rp_eq_fpm_pap_ann] 2022-01 (145/180, 81%) | 경과 1.9분 | ETA 0.5분
  [mat_rp_eq_fpm_pap_ann] 2023-01 (157/180, 87%) | 경과 2.1분 | ETA 0.3분
  [mat_rp_eq_fpm_pap_ann] 2024-01 (169/180, 94%) | 경과 2.2분 | ETA 0.1분
  [mat_rp_eq_fpm_pap_ann] 202

[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed: 16.6min


  [mat_eq_rp_inv_pap_ann] 2011-01 (13/180, 7%) | 경과 0.1분 | ETA 1.3분
  [mat_eq_rp_inv_pap_ann] 2012-01 (25/180, 14%) | 경과 0.3분 | ETA 1.9분
  [mat_eq_rp_inv_pap_ann] 2013-01 (37/180, 21%) | 경과 0.5분 | ETA 1.9분
  [mat_eq_rp_inv_pap_ann] 2014-01 (49/180, 27%) | 경과 0.7분 | ETA 1.9분
  [mat_eq_rp_inv_pap_ann] 2015-01 (61/180, 34%) | 경과 0.8분 | ETA 1.6분
  [mat_eq_rp_inv_pap_ann] 2016-01 (73/180, 41%) | 경과 1.0분 | ETA 1.5분
  [mat_eq_rp_inv_pap_ann] 2017-01 (85/180, 47%) | 경과 1.1분 | ETA 1.2분
  [mat_eq_rp_inv_pap_ann] 2018-01 (97/180, 54%) | 경과 1.2분 | ETA 1.0분
  [mat_eq_rp_inv_pap_ann] 2019-01 (109/180, 61%) | 경과 1.3분 | ETA 0.8분
  [mat_eq_rp_inv_pap_ann] 2020-01 (121/180, 67%) | 경과 1.4분 | ETA 0.7분
  [mat_eq_rp_inv_pap_ann] 2021-01 (133/180, 74%) | 경과 1.4분 | ETA 0.5분
  [mat_eq_rp_inv_pap_ann] 2022-01 (145/180, 81%) | 경과 1.6분 | ETA 0.4분
  [mat_eq_rp_inv_pap_ann] 2023-01 (157/180, 87%) | 경과 1.7분 | ETA 0.2분
  [mat_eq_rp_inv_pap_ann] 2024-01 (169/180, 94%) | 경과 1.8분 | ETA 0.1분
  [mat_eq_rp_inv_pap_ann] 202

[Parallel(n_jobs=1)]: Done  12 tasks      | elapsed: 27.1min


  [mat_rp_rp_fpm_pap] 2011-01 (13/180, 7%) | 경과 0.1분 | ETA 0.7분
  [mat_rp_rp_fpm_pap] 2012-01 (25/180, 14%) | 경과 0.1분 | ETA 0.7분
  [mat_rp_rp_fpm_pap] 2013-01 (37/180, 21%) | 경과 0.3분 | ETA 1.1분
  [mat_rp_rp_fpm_pap] 2014-01 (49/180, 27%) | 경과 0.4분 | ETA 1.2분
  [mat_rp_rp_fpm_pap] 2015-01 (61/180, 34%) | 경과 0.5분 | ETA 0.9분
  [mat_rp_rp_fpm_pap] 2016-01 (73/180, 41%) | 경과 0.7분 | ETA 1.0분
  [mat_rp_rp_fpm_pap] 2017-01 (85/180, 47%) | 경과 0.9분 | ETA 1.0분
  [mat_rp_rp_fpm_pap] 2018-01 (97/180, 54%) | 경과 1.0분 | ETA 0.9분
  [mat_rp_rp_fpm_pap] 2019-01 (109/180, 61%) | 경과 1.2분 | ETA 0.8분
  [mat_rp_rp_fpm_pap] 2020-01 (121/180, 67%) | 경과 1.4분 | ETA 0.7분
  [mat_rp_rp_fpm_pap] 2021-01 (133/180, 74%) | 경과 1.6분 | ETA 0.6분
  [mat_rp_rp_fpm_pap] 2022-01 (145/180, 81%) | 경과 1.7분 | ETA 0.4분
  [mat_rp_rp_fpm_pap] 2023-01 (157/180, 87%) | 경과 1.9분 | ETA 0.3분
  [mat_rp_rp_fpm_pap] 2024-01 (169/180, 94%) | 경과 2.0분 | ETA 0.1분
  [mat_rp_rp_fpm_pap] 2024-12 (180/180, 100%) | 경과 2.1분 | ETA 0.0분
  → [mat_rp_rp_fpm

[Parallel(n_jobs=1)]: Done  14 out of  14 | elapsed: 32.0min finished
